# R Project: Customer Churn Prediction
### ==========================
### 1. Load Libraries
### ==========================

In [ ]:
library(tidyverse)
library(caret)
library(randomForest)
library(xgboost)
library(pROC)
library(dplyr)


set.seed(42)

'data.frame':	7043 obs. of  21 variables:
 $ customerID      : chr  "7590-VHVEG" "5575-GNVDE" "3668-QPYBK" "7795-CFOCW" ...
 $ gender          : chr  "Female" "Male" "Male" "Male" ...
 $ SeniorCitizen   : int  0 0 0 0 0 0 0 0 0 0 ...
 $ Partner         : chr  "Yes" "No" "No" "No" ...
 $ Dependents      : chr  "No" "No" "No" "No" ...
 $ tenure          : int  1 34 2 45 2 8 22 10 28 62 ...
 $ PhoneService    : chr  "No" "Yes" "Yes" "No" ...
 $ MultipleLines   : chr  "No phone service" "No" "No" "No phone service" ...
 $ InternetService : chr  "DSL" "DSL" "DSL" "DSL" ...
 $ OnlineSecurity  : chr  "No" "Yes" "Yes" "Yes" ...
 $ OnlineBackup    : chr  "Yes" "No" "Yes" "No" ...
 $ DeviceProtection: chr  "No" "Yes" "No" "Yes" ...
 $ TechSupport     : chr  "No" "No" "No" "Yes" ...
 $ StreamingTV     : chr  "No" "No" "No" "No" ...
 $ StreamingMovies : chr  "No" "No" "No" "No" ...
 $ Contract        : chr  "Month-to-month" "One year" "Month-to-month" "One year" ...
 $ PaperlessBilling: chr  "Yes"

### ==========================
### 2. Load Data & Initial Preprocessing
### ==========================

In [46]:
df <- read.csv("telco_customer_churn.csv")

In [48]:
# Check for missing values
cat("Missing values in dataset:\n")
print(colSums(is.na(df)))

Missing values in dataset:


      customerID           gender    SeniorCitizen          Partner 
               0                0                0                0 
      Dependents           tenure     PhoneService    MultipleLines 
               0                0                0                0 
 InternetService   OnlineSecurity     OnlineBackup DeviceProtection 
               0                0                0                0 
     TechSupport      StreamingTV  StreamingMovies         Contract 
               0                0                0                0 
PaperlessBilling    PaymentMethod   MonthlyCharges     TotalCharges 
               0                0                0               11 
           Churn 
               0 


In [49]:
# Drop customerID (not useful for prediction)
df <- df %>% select(-customerID)

In [50]:
# Handle TotalCharges if it's character with empty strings
if(is.character(df$TotalCharges)) {
  df$TotalCharges[df$TotalCharges == ""] <- NA
  df$TotalCharges <- as.numeric(df$TotalCharges)
}

In [51]:
# Convert Churn to factor (binary outcome)
df$Churn <- ifelse(df$Churn == "Yes", 1, 0)

In [52]:
# Convert character predictors to factors
df <- df %>% mutate_if(is.character, as.factor)

### ==========================
### 3. IMPUTE MISSING VALUES (PUT IT HERE!)
### ==========================

In [53]:
# For numeric variables, impute with median
numeric_cols <- sapply(df, is.numeric)
for(col in names(df)[numeric_cols]) {
  if(sum(is.na(df[[col]])) > 0) {
    df[[col]][is.na(df[[col]])] <- median(df[[col]], na.rm = TRUE)
  }
}

In [54]:
# For factor variables, impute with mode
factor_cols <- sapply(df, is.factor)
for(col in names(df)[factor_cols]) {
  if(sum(is.na(df[[col]])) > 0) {
    mode_val <- names(sort(table(df[[col]]), decreasing = TRUE))[1]
    df[[col]][is.na(df[[col]])] <- mode_val
  }
}

In [55]:
# Final check for missing values
cat("Missing values after imputation:\n")
print(colSums(is.na(df)))

Missing values after imputation:
          gender    SeniorCitizen          Partner       Dependents 
               0                0                0                0 
          tenure     PhoneService    MultipleLines  InternetService 
               0                0                0                0 
  OnlineSecurity     OnlineBackup DeviceProtection      TechSupport 
               0                0                0                0 
     StreamingTV  StreamingMovies         Contract PaperlessBilling 
               0                0                0                0 
   PaymentMethod   MonthlyCharges     TotalCharges            Churn 
               0                0                0                0 


### ==========================
### 4. Train/Test Split
### ==========================

In [56]:
trainIndex <- createDataPartition(df$Churn, p = 0.8, list = FALSE)
train <- df[trainIndex, ]
test  <- df[-trainIndex, ]

train_label <- train$Churn
test_label  <- test$Churn

### ==========================
### 5. Logistic Regression
### ==========================

In [57]:
logit_model <- glm(Churn ~ ., data = train, family = binomial)

logit_probs <- predict(logit_model, newdata = test, type = "response")
logit_preds <- ifelse(logit_probs > 0.5, 1, 0)

cat("\n--- Logistic Regression ---\n")
print(confusionMatrix(as.factor(logit_preds), as.factor(test_label), positive = "1"))
roc_logit <- roc(test_label, logit_probs)
cat("AUC (Logistic Regression):", auc(roc_logit), "\n")


--- Logistic Regression ---
Confusion Matrix and Statistics

          Reference
Prediction   0   1
         0 943 173
         1 101 191
                                          
               Accuracy : 0.8054          
                 95% CI : (0.7837, 0.8258)
    No Information Rate : 0.7415          
    P-Value [Acc > NIR] : 1.011e-08       
                                          
                  Kappa : 0.4575          
                                          
 Mcnemar's Test P-Value : 1.793e-05       
                                          
            Sensitivity : 0.5247          
            Specificity : 0.9033          
         Pos Pred Value : 0.6541          
         Neg Pred Value : 0.8450          
             Prevalence : 0.2585          
         Detection Rate : 0.1357          
   Detection Prevalence : 0.2074          
      Balanced Accuracy : 0.7140          
                                          
       'Positive' Class : 1               
 

Setting levels: control = 0, case = 1

Setting direction: controls < cases



AUC (Logistic Regression): 0.8562455 


### ==========================
### 6. Random Forest
### ==========================

In [58]:
rf_model <- randomForest(as.factor(Churn) ~ ., data = train, ntree = 200, mtry = 5)

rf_preds <- predict(rf_model, newdata = test, type = "response")
rf_probs <- predict(rf_model, newdata = test, type = "prob")[,2]

cat("\n--- Random Forest ---\n")
print(confusionMatrix(rf_preds, as.factor(test_label), positive = "1"))
roc_rf <- roc(test_label, rf_probs)
cat("AUC (Random Forest):", auc(roc_rf), "\n")


--- Random Forest ---
Confusion Matrix and Statistics

          Reference
Prediction   0   1
         0 942 173
         1 102 191
                                         
               Accuracy : 0.8047         
                 95% CI : (0.783, 0.8251)
    No Information Rate : 0.7415         
    P-Value [Acc > NIR] : 1.468e-08      
                                         
                  Kappa : 0.456          
                                         
 Mcnemar's Test P-Value : 2.430e-05      
                                         
            Sensitivity : 0.5247         
            Specificity : 0.9023         
         Pos Pred Value : 0.6519         
         Neg Pred Value : 0.8448         
             Prevalence : 0.2585         
         Detection Rate : 0.1357         
   Detection Prevalence : 0.2081         
      Balanced Accuracy : 0.7135         
                                         
       'Positive' Class : 1              
                           

Setting levels: control = 0, case = 1

Setting direction: controls < cases



AUC (Random Forest): 0.8488722 


### ==========================
### 7. XGBoost
### ==========================

In [59]:
# Prepare matrices
train_matrix <- model.matrix(Churn ~ . , data = train)[, -1]
test_matrix  <- model.matrix(Churn ~ . , data = test)[, -1]

dtrain <- xgb.DMatrix(data = train_matrix, label = train_label)
dtest  <- xgb.DMatrix(data = test_matrix,  label = test_label)

params <- list(
  objective = "binary:logistic",
  eval_metric = "auc"
)

xgb_model <- xgb.train(
  params = params,
  data = dtrain,
  nrounds = 100,
  watchlist = list(train = dtrain, test = dtest),
  verbose = 0
)

xgb_probs <- predict(xgb_model, dtest)
xgb_preds <- ifelse(xgb_probs > 0.5, 1, 0)

cat("\n--- XGBoost ---\n")
print(confusionMatrix(as.factor(xgb_preds), as.factor(test_label), positive = "1"))
roc_xgb <- roc(test_label, xgb_probs)
cat("AUC (XGBoost):", auc(roc_xgb), "\n")

Warning message in throw_err_or_depr_msg("Parameter '", match_old, "' has been renamed to '", :
"Parameter 'watchlist' has been renamed to 'evals'. This warning will become an error in a future version."



--- XGBoost ---
Confusion Matrix and Statistics

          Reference
Prediction   0   1
         0 917 178
         1 127 186
                                          
               Accuracy : 0.7834          
                 95% CI : (0.7609, 0.8046)
    No Information Rate : 0.7415          
    P-Value [Acc > NIR] : 0.0001452       
                                          
                  Kappa : 0.408           
                                          
 Mcnemar's Test P-Value : 0.0041966       
                                          
            Sensitivity : 0.5110          
            Specificity : 0.8784          
         Pos Pred Value : 0.5942          
         Neg Pred Value : 0.8374          
             Prevalence : 0.2585          
         Detection Rate : 0.1321          
   Detection Prevalence : 0.2223          
      Balanced Accuracy : 0.6947          
                                          
       'Positive' Class : 1               
             

Setting levels: control = 0, case = 1

Setting direction: controls < cases



AUC (XGBoost): 0.8408949 


### ==========================
### 8. Model Comparison
### ==========================

In [60]:
auc_results <- data.frame(
  Model = c("Logistic Regression", "Random Forest", "XGBoost"),
  AUC   = c(auc(roc_logit), auc(roc_rf), auc(roc_xgb))
)

print(auc_results)

                Model       AUC
1 Logistic Regression 0.8562455
2       Random Forest 0.8488722
3             XGBoost 0.8408949
